# Phase 2b: Spatial feature extraction for attention

The original `02_features.ipynb` cached ResNet-50's *pooled* output -- one 2048-d vector per image, everything spatial averaged away. Attention needs the opposite: features that still have spatial structure, so the model can learn "which region matters for this word."

ResNet-50's `layer4` output (before the final average-pool + fc) is a `(2048, 7, 7)` feature map for a 224x224 input -- 49 spatial locations, each described by a 2048-d vector. That's what we cache here instead.

In [ ]:
from pathlib import Path
import pandas as pd
import pickle
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from tqdm import tqdm

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
IMAGES_DIR = DATA_DIR / "Images"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

df = pd.concat([
    pd.read_csv(ARTIFACTS_DIR / "train.csv"),
    pd.read_csv(ARTIFACTS_DIR / "val.csv"),
    pd.read_csv(ARTIFACTS_DIR / "test.csv"),
], ignore_index=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# Same frozen ResNet-50 as before, but stop BEFORE the average pool + fc.
# nn.Sequential(*list(resnet.children())[:-2]) keeps every conv block
# (through layer4) and drops avgpool + fc -- output is (batch, 2048, 7, 7)
# instead of (batch, 2048).
weights = ResNet50_Weights.DEFAULT
resnet = resnet50(weights=weights)
cnn = nn.Sequential(*list(resnet.children())[:-2])
cnn = cnn.to(device)
cnn.eval()  # same .eval() lesson as before -- BatchNorm must not use live batch stats

tfm = weights.transforms()

# sanity check the output shape before running it over all 8091 images
dummy = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    out = cnn(dummy)
print(out.shape)  # expect torch.Size([1, 2048, 7, 7])

In [ ]:
class ImageOnlyDataset(Dataset):
    def __init__(self, filenames, images_dir, transform):
        self.filenames = filenames
        self.images_dir = images_dir
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img = Image.open(self.images_dir / fname).convert("RGB")
        return fname, self.transform(img)

all_filenames = df["image"].unique().tolist()
img_dataset = ImageOnlyDataset(all_filenames, IMAGES_DIR, tfm)
# num_workers=0 -- the Windows/Jupyter DataLoader hang from 02_features.ipynb
img_loader = DataLoader(img_dataset, batch_size=32, shuffle=False, num_workers=0)
print(len(img_dataset), "images to process")

In [ ]:
filenames_out = []
features_list = []

with torch.no_grad():
    for fnames, imgs in tqdm(img_loader):
        imgs = imgs.to(device)
        out = cnn(imgs)  # (batch, 2048, 7, 7)

        # reshape to (batch, 49, 2048) -- 49 spatial "locations", each a
        # 2048-d description. This is the shape the attention module will
        # expect: one row per region to attend over.
        out = out.flatten(2).permute(0, 2, 1)  # (batch, 49, 2048)
        out = out.half().cpu()

        for fname, vec in zip(fnames, out):
            filenames_out.append(fname)
            features_list.append(vec.clone())  # .clone() -- same storage-aliasing bug as last time

feature_matrix = torch.stack(features_list)  # (8091, 49, 2048)
print(feature_matrix.shape, feature_matrix.dtype)

In [ ]:
with open(ARTIFACTS_DIR / "features_spatial.pkl", "wb") as f:
    pickle.dump({"filenames": filenames_out, "matrix": feature_matrix}, f)

import os
size_mb = os.path.getsize(ARTIFACTS_DIR / "features_spatial.pkl") / 1024**2
print(f"saved {len(filenames_out)} spatial features, {size_mb:.1f} MB")
# expect roughly 8091 * 49 * 2048 * 2 bytes ~= 1.55 GB -- much bigger than
# the 32 MB flat-feature cache, since we kept all 49 regions instead of
# averaging them into one vector.